# 🌱 RAG Híbrido y Chat SLM (FarmifAI)

Este notebook implementa:
1. **RAG Híbrido** (BM25 + Semántica + Reranker) sobre tu base de conocimientos.
2. **Descarga Automática del Modelo SLM** (desde Hugging Face con conversión GGUF si es necesario).
3. **Interfaz de Chat Interactiva (Gradio)** para probar el modelo localmente usando `llama.cpp`.

## 1. Instalación de Dependencias
Instalaremos las dependencias necesarias. Se intenta compilar `llama-cpp-python` con soporte para GPU (CUDA) para máxima velocidad en Colab.

In [1]:
# 1. Instalar librerías para RAG y UI de forma silenciosa
!pip install -q rank-bm25 sentence-transformers huggingface_hub gradio

# 2. Asegurar una versión de numpy compatible con el entorno de Colab
!pip install -q "numpy>=1.22,<2.1"

# 3. Instalar el binario precompilado de llama-cpp-python para CUDA 12
!pip install -q llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 \
  --upgrade --no-cache-dir

## 2. Inicialización y Carga de Datos (RAG)
Configuraremos las funciones para cargar los chunks, procesar el texto y generar embeddings.

In [2]:
import json
import os
import pickle
import torch
import re
import time
import unicodedata
from typing import Any
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

try:
    from google.colab import files as colab_files
    IN_COLAB = True
    print("✅ Ejecutando en Google Colab")
except ImportError:
    IN_COLAB = False
    print("ℹ️ Ejecutando en entorno local")

CHUNKS_FILENAME = "knowledge_base.json"
EMBEDDINGS_FILENAME = "embeddings_chunks.pkl"
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-base"

def load_chunks(filename: str) -> list[dict]:
    if not os.path.exists(filename):
        if IN_COLAB:
            print(f"📂 Sube el archivo '{filename}':")
            uploaded = colab_files.upload()
            if filename not in uploaded:
                raise FileNotFoundError(f"No se subió '{filename}'.")
        else:
            raise FileNotFoundError(f"No se encontró '{filename}'.")
    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data["chunks"]

def preprocess_spanish(text: str) -> list[str]:
    text = text.lower()
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return [t for t in text.split() if len(t) > 1]

def load_or_generate_embeddings(chunks, model_name, embeddings_file, batch_size=128):
    if torch.cuda.is_available():
      torch.cuda.empty_cache()

    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = SentenceTransformer(model_name, device=device)
    embeddings = None
    if os.path.exists(embeddings_file):
        with open(embeddings_file, "rb") as f:
            embeddings = pickle.load(f)
    elif IN_COLAB:
        print(f"\n📂 Sube '{embeddings_file}' si lo tienes (o cancela para generar):")
        try:
            uploaded = colab_files.upload()
            if embeddings_file in uploaded:
                with open(embeddings_file, "rb") as f:
                    embeddings = pickle.load(f)
        except Exception:
            pass

    if embeddings is None or embeddings.shape[0] != len(chunks):
        print("\n⏳ Generando embeddings...")
        texts = [chunk["text"] for chunk in chunks]
        embeddings = model.encode(texts, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
        with open(embeddings_file, "wb") as f:
            pickle.dump(embeddings, f)
        if IN_COLAB:
            colab_files.download(embeddings_file)
    # Movemos el modelo a CPU para liberar VRAM y evitar conflictos de memoria con llama-cpp
    model.to('cpu')
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return embeddings, model

class HybridRAGPipeline:
    def __init__(self, chunks, bm25_index, corpus_tokens, chunk_embeddings, embedding_model, reranker_model_name=RERANKER_MODEL_NAME):
        self.chunks = chunks
        self.bm25 = bm25_index
        self.corpus_tokens = corpus_tokens
        self.chunk_embeddings = chunk_embeddings
        self.embedding_model = embedding_model

        # Instanciamos el reranker en CPU para evitar conflicto de acceso ilegal a memoria con llama.cpp
        device = "cpu"

        self.reranker = CrossEncoder(reranker_model_name, device=device)

    def search_bm25(self, query: str, top_k: int = 10):
        scores = self.bm25.get_scores(preprocess_spanish(query))
        return [(int(i), float(scores[i])) for i in np.argsort(scores)[::-1][:top_k]]

    def search_semantic(self, query: str, top_k: int = 10):
        query_embedding = self.embedding_model.encode(query, normalize_embeddings=True, convert_to_numpy=True)
        similarities = self.chunk_embeddings @ query_embedding
        return [(int(i), float(similarities[i])) for i in np.argsort(similarities)[::-1][:top_k]]

    @staticmethod
    def reciprocal_rank_fusion(results_lists, k=60):
        rrf_scores = {}
        for results in results_lists:
            for rank, (idx, _) in enumerate(results):
                rrf_scores[idx] = rrf_scores.get(idx, 0.0) + 1.0 / (k + rank + 1)
        return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    def rerank(self, query: str, candidate_indices: list[int]):
        pairs = [(query, self.chunks[i]["text"]) for i in candidate_indices]
        scores = self.reranker.predict(pairs)
        scored = list(zip(candidate_indices, [float(s) for s in scores]))
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored

    def query(self, user_query: str, top_k_lexical=10, top_k_semantic=10, top_k_final=3):
        bm25_res = self.search_bm25(user_query, top_k=top_k_lexical)
        sem_res = self.search_semantic(user_query, top_k=top_k_semantic)
        hybrid_res = self.reciprocal_rank_fusion([bm25_res, sem_res])
        candidate_indices = [idx for idx, _ in hybrid_res]
        reranked = self.rerank(user_query, candidate_indices)
        return [{**self.chunks[idx], "rerank_score": score, "_index": idx} for idx, score in reranked[:top_k_final]]

# Inicialización (Ejecutar solo tras subir knowledge_base.json)
try:
    chunks = load_chunks(CHUNKS_FILENAME)
    corpus_tokens = [preprocess_spanish(c["text"]) for c in chunks]
    bm25 = BM25Okapi(corpus_tokens)
    chunk_embeddings, embedding_model = load_or_generate_embeddings(chunks, EMBEDDING_MODEL_NAME, EMBEDDINGS_FILENAME)
    rag = HybridRAGPipeline(chunks, bm25, corpus_tokens, chunk_embeddings, embedding_model)
    print("\n✅ RAG Inicializado correctamente.")
except Exception as e:
    print(f"⚠️ Error inicializando RAG (¿Falta subir knowledge_base.json?): {e}")

✅ Ejecutando en Google Colab


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


✅ RAG Inicializado correctamente.


## 3. Descarga del Modelo SLM
Descargamos `FarmifAI/Qwen3.5-0.8B_FarmifAI2.0`.
Si no tiene archivo GGUF, descargará los pesos de Hugging Face y los convertirá.

In [3]:
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

model_repo = "FarmifAI/Qwen3.5-0.8B_FarmifAI2.0"
gguf_path = "model.gguf"

try:
    api = HfApi()
    repo_files = api.list_repo_files(model_repo)
    gguf_files = [f for f in repo_files if f.endswith("Q4_K_M.gguf")]

    if gguf_files:
        target_file = gguf_files[0]
        print(f"✅ Se encontró GGUF en el repositorio. Descargando {target_file}...")
        gguf_path = hf_hub_download(repo_id=model_repo, filename=target_file, local_dir=".")
        print(f"✅ Descargado en: {gguf_path}")
    else:
        print("⚠️ No se encontró GGUF nativo. Descargando repositorio completo para conversión...")
        if not os.path.exists("llama.cpp"):
            !git clone https://github.com/ggml-org/llama.cpp
            !pip install -r llama.cpp/requirements.txt -q

        hf_model_dir = snapshot_download(repo_id=model_repo, local_dir="hf_model_dir")
        print("⏳ Convirtiendo el modelo a GGUF (f16)...")
        !python llama.cpp/convert_hf_to_gguf.py hf_model_dir --outfile model.gguf --outtype f16
        gguf_path = "model.gguf"
        print(f"✅ Conversión finalizada: {gguf_path}")
except Exception as e:
    print(f"❌ Error al obtener modelo: {e}")
    print("🔄 Usando modelo alternativo Qwen 0.5B como fallback...")
    fallback_repo = "Qwen/Qwen1.5-0.5B-Chat-GGUF"
    fallback_file = "qwen1_5-0_5b-chat-q4_k_m.gguf"
    gguf_path = hf_hub_download(repo_id=fallback_repo, filename=fallback_file, local_dir=".")
    print(f"✅ Fallback descargado: {gguf_path}")

✅ Se encontró GGUF en el repositorio. Descargando Qwen3.5-0.8B.Q4_K_M.gguf...
✅ Descargado en: /content/Qwen3.5-0.8B.Q4_K_M.gguf


## 4. Inicialización de llama-cpp
Cargamos el modelo GGUF usando `llama-cpp-python`.

In [4]:
from llama_cpp import Llama

print(f"⏳ Cargando modelo desde {gguf_path}...")
llm = Llama(
    model_path=gguf_path,
    n_gpu_layers=-1, # Usar GPU si está disponible
    n_ctx=4096,      # Ventana de contexto para soportar RAG chunks
    verbose=False    # Ocultar logs detallados de C++
)
print("✅ Modelo cargado con éxito en llama-cpp.")

⏳ Cargando modelo desde /content/Qwen3.5-0.8B.Q4_K_M.gguf...


llama_context: n_ctx_seq (4096) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


✅ Modelo cargado con éxito en llama-cpp.


In [8]:
print(rag.query("condiciones de suelo para el cacao"))

AcceleratorError: CUDA error: an illegal memory access was encountered
Search for `cudaErrorIllegalAddress' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## 5. Interfaz de Chat (Gradio UI)
Esta celda levanta la interfaz gráfica. Ofrece una vista amigable del chat y pestañas laterales para revisar el JSON de la petición y los chunks del RAG.

In [6]:
import gradio as gr

def parse_assistant_response(response_text: str):
    """Extrae el razonamiento y la respuesta de las etiquetas."""
    reasoning_match = re.search(r'<reasoning>(.*?)</reasoning>', response_text, re.DOTALL)
    reasoning = reasoning_match.group(1).strip() if reasoning_match else ""

    answer_match = re.search(r'<answer>(.*?)</answer>', response_text, re.DOTALL)
    if answer_match:
        answer = answer_match.group(1).strip()
    else:
        # Fallback
        answer = re.sub(r'<reasoning>.*?</reasoning>', '', response_text, flags=re.DOTALL).strip()
        answer = answer.replace('<answer>', '').replace('</answer>', '').strip()

    return reasoning, answer

def respond(message, history):
    if not message.strip():
        return "", history, {}, "*Escribe una pregunta para ver el razonamiento.*", []

    # 1. Recuperar chunks del RAG
    try:
        rag_results = rag.query(message, top_k_lexical=10, top_k_semantic=10, top_k_final=3)
        knowledge_text = "\n\n".join([
            f"Documento: {res['document_id']} (Fragmento {res['chunk_number']})\nContenido: {res['text']}"
            for res in rag_results
        ])
    except Exception as e:
        print(f"RAG No inicializado: {e}")
        rag_results = []
        knowledge_text = "No hay contexto disponible."

    # 2. Construir Prompts
    system_prompt = (
        "Eres un asistente inteligente. Analiza la información proporcionada en la etiqueta <knowledge> para responder a la solicitud del usuario. Responde estrictamente en el siguiente formato:\n"
        "<reasoning>\n...\n</reasoning>\n<answer>\n...\n</answer>"
    )

    user_prompt = f"<knowledge>\n{knowledge_text}\n</knowledge>\n\n{message}"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    # 3. Llamar al modelo LLM
    response = llm.create_chat_completion(
        messages=messages,
        temperature=0.1,
        max_tokens=1024,
    )

    raw_response = response['choices'][0]['message']['content']
    reasoning, answer = parse_assistant_response(raw_response)

    if not answer:
        answer = raw_response

    # 4. Formatear salidas de Debug / Inspección
    query_json = {
        "messages_sent": messages,
        "raw_response": raw_response
    }

    rag_chunks = [
        {
            "document_id": res.get("document_id", ""),
            "chunk_number": res.get("chunk_number", ""),
            "rerank_score": float(res.get("rerank_score", 0.0)),
            "text": res.get("text", "")
        }
        for res in rag_results
    ]

    history.append((message, answer))
    reasoning_md = f"```markdown\n{reasoning}\n```" if reasoning else "*No se extrajo etiqueta <reasoning>.*"

    return "", history, query_json, reasoning_md, rag_chunks

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🌱 FarmifAI Agro-Asistente RAG")

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=550, label="Conversación con FarmifAI")
            msg = gr.Textbox(label="Ingresa tu pregunta agrícola...", placeholder="Ej. ¿Cuáles son las condiciones de suelo para el cacao?")
            with gr.Row():
                send_btn = gr.Button("Enviar", variant="primary")
                clear_btn = gr.ClearButton([chatbot, msg], value="Limpiar Historial")

        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.Tab("Detalles de la Consulta"):
                    gr.Markdown("### Prompt enviado y respuesta cruda")
                    query_json_viewer = gr.JSON(label="JSON")
                with gr.Tab("Razonamiento"):
                    gr.Markdown("### Razonamiento Interno (<reasoning>)")
                    reasoning_viewer = gr.Markdown(value="*El razonamiento aparecerá aquí.*")
                with gr.Tab("Fragmentos RAG"):
                    gr.Markdown("### Chunks inyectados en <knowledge>")
                    chunks_viewer = gr.JSON(label="Chunks")

    # Eventos
    msg.submit(respond, [msg, chatbot], [msg, chatbot, query_json_viewer, reasoning_viewer, chunks_viewer])
    send_btn.click(respond, [msg, chatbot], [msg, chatbot, query_json_viewer, reasoning_viewer, chunks_viewer])

demo.launch(share=True, inline=True)


/tmp/ipykernel_3335/669517377.py:80: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_3335/669517377.py:85: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=550, label="Conversación con FarmifAI")
/tmp/ipykernel_3335/669517377.py:85: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=550, label="Conversación con FarmifAI")


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://74c1f0867687e0dc99.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
